In [0]:
%run ../control_framework/__init__ 

In [0]:
import time
from pyspark.sql import Row

In [0]:
dbutils.widgets.text("env", "dev", "env")
# dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "finnhub_daily_ingestion", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")
dbutils.widgets.text("run_type", "load", "run type")
dbutils.widgets.text("source", "finnhub", "Source")
dbutils.widgets.text("target", "m101_finnhub", "Target")
dbutils.widgets.text("execution_type", "ingestion", "execution type")

In [0]:
env = dbutils.widgets.get("env")
# param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
run_type = dbutils.widgets.get("run_type")
source = dbutils.widgets.get("source")
target = dbutils.widgets.get("target")
execution_type = dbutils.widgets.get("execution_type")
print(f"env: {env}")
# print(f"param_name: {param_name}")
print(f"process_name: {process_name}")
print(f"process_date: {process_date}")
print(f"run_type: {run_type}")
print(f"source: {source}")
print(f"target: {target}")
print(f"execution_type: {execution_type}")

In [0]:
try:
    comment = register_tgt(target, execution_type)
    insert_log(target, process_name, "source_register","source_register","source_register","source_register",process_date, 'success',comment)
except Exception as e:
    error_msg = str(e).replace("'", "''")
    insert_log(target, process_name, "source_register","source_register","source_register","source_register",process_date, 'failed', error_msg)
    print(f"Error in register_source: {e}")
    dbutils.notebook.exit(f"Error in register_source: {e}")

In [0]:
src_schema1 = f_get_src_schema(source)
src_tbl1 = ''
src_schema_master = f_get_src_schema('master')
src_master_tb1 = 's_and_p_500_dev'
target_schema = f_get_tgt_schema(target)
target_tbl = 'finnhubb_tb1_ts'
print(src_schema1)
print(src_tbl1)
print(src_schema_master)
print(src_master_tb1)
print(target_schema)
print(target_tbl)


In [0]:
src_tbl_list = [{"source":source,"source_schema":src_schema1,"source_table": src_tbl1},
                {"source":'master',"source_schema":src_schema_master,"source_table": src_master_tb1}
                ]
tgt_tbl_list = [{"target":target,"target_schema":target_schema,"target_table": target_tbl,'holiday_ind': 'N','weekend_ind': 'N','calender':'','frequency': 'daily'}]

In [0]:
# {'c': 333.74, 'd': 0.48, 'dp': 0.144, 'h': 334.99, 'l': 329.0006, 'o': 331.98, 'pc': 333.26, 't': 1784318400}

In [0]:
src_tgt_mapping_list = [
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'c','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'c','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'d','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'d','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'dp','source_column_datatype':'decimal(20,3)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'dp','target_column_datatype':'decimal(20,3)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'h','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'h','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'l','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'l','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'o','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'o','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'pc','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'pc','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'t','source_column_datatype':'bigint','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'t','target_column_datatype':'bigint'},
    {'source_schema':src_schema_master,'source_table':src_master_tb1,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'symbol','target_column_datatype':'string'}
]

In [0]:
if run_type == 'load_metadata':
    status = run_load_metadata(src_tbl_list,tgt_tbl_list,target,process_name,src_tgt_mapping_list,process_date)
    dbutils.notebook.exit(status)


In [0]:
if run_type == 'full':
    create_tgt_tbl_log = create_tgt_tbl(tgt_tbl_list)
    if  create_tgt_tbl_log == f"Table {target_schema}.{target_tbl} created successfully":
        process_status = 'success'
    else:
        process_status = 'failed'
    log = str(create_tgt_tbl_log).replace("'", "''")
    insert_log (target, process_name, 'create_tgt_tbl','create_tgt_tbl', target_schema, target_tbl, process_date,process_status, log)

In [0]:
def get_symbol_list(src_schema_master,src_master_tb1):
    query = f"select symbol from {src_schema_master}.{src_master_tb1}"
    print(query)
    symbol_df = spark.sql(query).toPandas()
    symbol_list = symbol_df['symbol'].tolist()
    return symbol_list


In [0]:
symbol_list = get_symbol_list(src_schema_master,src_master_tb1)

In [0]:
def data_list(symbol_list):
    symbol_count = len(symbol_list)
    data_list = []
    for i in range(symbol_count):
        if i % 50 == 0 and i != 0:
            time.sleep(60)  # add one minute buffer
        data = get_finnhub_connection(symbol_list[i])
        data['symbol'] = symbol_list[i]
        data['process_date'] = process_date
        data['r_source'] = source
        # print(data)
        hash_id = create_hash_id(data)
        data['hash_id'] = hash_id
        # print(data)
        data_list.append(data)
    return data_list
# data_list = data_list(['MMM','ABT'])
# print(data_list)

In [0]:
# def insert_ingestion_table_query(data_list, target_schema, target_tbl):
#     column_types = {
#         'c': 'decimal(20,2)',
#         'd': 'decimal(20,2)',
#         'dp': 'decimal(20,3)',
#         'h': 'decimal(20,2)',
#         'l': 'decimal(20,2)',
#         'o': 'decimal(20,2)',
#         'pc': 'decimal(20,2)',
#         't': 'bigint',
#         'symbol': 'string',
#         'process_date': 'string',
#         'r_source': 'string',
#         'hash_id': 'string'
#     }
#     try:
#         rows = [Row(**item) for item in data_list]
#         df = spark.createDataFrame(rows)
#         df.createOrReplaceTempView("temp_ingest_tbl")
#         select_expr = ", ".join([f"CAST({col} AS {dtype}) AS {col}" for col, dtype in column_types.items()])
#         query = f"""
#             INSERT INTO {target_schema}.{target_tbl} (c, d, dp, h, l, o, pc, t, symbol, process_date, r_source, hash_id)
#             SELECT {select_expr} FROM temp_ingest_tbl where hash_id not in (select hash_id from {target_schema}.{target_tbl})
#         """
#         spark.sql(query)
#         insert_log(target, process_name, 'insert_ingestion_table_query', 'insert_ingestion_table_query', target_schema, target_tbl, process_date, 'success', 'Ingestion completed')
#         spark.sql(f"DROP TABLE IF EXISTS temp_ingest_tbl")
#     except Exception as e:
#         error_msg = str(e).replace("'", "''")
#         insert_log(target, process_name, 'insert_ingestion_table_query', 'insert_ingestion_table_query', target_schema, target_tbl, process_date, 'failed', error_msg)
#         print(f"Error in insert_ingestion_table_query: {e}")
#         dbutils.notebook.exit(f"Error in insert_ingestion_table_query: {e}")

# # insert_ingestion_table_query(data_list, target_schema, target_tbl)

In [0]:
def insert_ingestion_table_query(data_list, target_schema, target_tbl):
    column_types = {
        'c': 'decimal(20,2)',
        'd': 'decimal(20,2)',
        'dp': 'decimal(20,3)',
        'h': 'decimal(20,2)',
        'l': 'decimal(20,2)',
        'o': 'decimal(20,2)',
        'pc': 'decimal(20,2)',
        't': 'bigint',
        'symbol': 'string',
        'process_date': 'string',
        'r_source': 'string',
        'hash_id': 'string'
    }
    column_list = list(column_types.keys())
    # Create string table if not exists
    create_table_columns = ", ".join([f"{col} string" for col in column_list])
    
    create_table_query = f"CREATE TABLE IF NOT EXISTS {target_schema}.string_{target_tbl} ({create_table_columns}) USING DELTA PARTITIONED BY (process_date)"
    spark.sql(create_table_query)

    failed_rows = []
    for item in data_list:
        try:
            hash_id = str(item.get('hash_id', ''))
            check_query = f"SELECT 1 FROM {target_schema}.{target_tbl} WHERE hash_id = '{hash_id}' LIMIT 1"
            exists = spark.sql(check_query).count() > 0
            if not exists:
                values = ", ".join([f"CAST('{str(item.get(col, ''))}' AS {column_types[col]})" for col in column_list])
                insert_target_query = f"""
                    INSERT INTO {target_schema}.{target_tbl} (c, d, dp, h, l, o, pc, t, symbol, process_date, r_source, hash_id)
                    VALUES ({values})
                """
                spark.sql(insert_target_query)
        except Exception as e:
            failed_rows.append({col: str(item.get(col, "")) for col in column_list})

    # Insert failed rows into string table
    if failed_rows:
        from pyspark.sql import Row
        df_string = spark.createDataFrame([Row(**row) for row in failed_rows])
        df_string.createOrReplaceTempView("temp_string_tbl")
        insert_string_query = f"""
            INSERT INTO {target_schema}.string_{target_tbl} (c, d, dp, h, l, o, pc, t, symbol, process_date, r_source, hash_id)
            SELECT c, d, dp, h, l, o, pc, t, symbol, process_date, r_source, hash_id FROM temp_string_tbl
        """
        spark.sql(insert_string_query)
        spark.sql("DROP VIEW IF EXISTS temp_string_tbl")

    insert_log(target, process_name, 'insert_ingestion_table_query', 'insert_ingestion_table_query', target_schema, target_tbl, process_date, 'success', 'Ingestion completed')

In [0]:
if run_type == "rollback":
    try:
        comment = rollback_run(tgt_tbl_list,process_date)
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'success', str(comment).replace("'", "''"))
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'failed', error_msg)
        comment = f"Error in rollback: {e}"
    dbutils.notebook.exit(f"{comment}")

In [0]:
if run_type == 'full' or run_type == 'load':
    try:
        data_list_val = data_list(symbol_list)
        insert_ingestion_table_query(data_list_val, target_schema, target_tbl)
        insert_log(target, process_name, 'data_ingestion', 'data_ingestion', target_schema, target_tbl, process_date, 'success', 'Data ingestion completed')
        e = "notebook run successful"
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'data_ingestion', 'data_ingestion', target_schema, target_tbl, process_date, 'failed', error_msg)
        e = f"Error in data_ingestion: {e}"
    dbutils.notebook.exit(f"{e}")